# Import Driver Route Data and Takeover Events

This notebook is intentionally small and incremental. It:

1. Discovers the `driver_*/route_*` folders that are present locally.
2. Imports every route CSV into a nested `routes` dictionary.
3. Builds concatenated per-modality DataFrames in `data_by_modality`.
4. Imports benchmark takeover events.
5. Filters takeover events to only the routes present in `dataset/`.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## Paths and Route Files

If you launch Jupyter from the project root, this works as-is. If you launch it from `notebooks/`, the root is adjusted automatically.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / "dataset"
BENCHMARK_DIR = DATASET_ROOT / "benchmark"
TAKEOVER_EVENTS_PATH = BENCHMARK_DIR / "takeover_events_or.csv"

ROUTE_FILES = {
    "gps": "gps.csv",
    "driver_state": "driver_state.csv",
    "radar": "radar.csv",
    "imu": "imu.csv",
    "localization": "localization.csv",
    "vehicle_dynamics": "vehicle_dynamics.csv",
    "planning": "planning.csv",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Takeover events: {TAKEOVER_EVENTS_PATH}")

Project root: /Users/mbaeuerl/coding/ComputationalModelsSoSe26
Dataset root: /Users/mbaeuerl/coding/ComputationalModelsSoSe26/dataset
Takeover events: /Users/mbaeuerl/coding/ComputationalModelsSoSe26/dataset/benchmark/takeover_events_or.csv


## Discover Present Driver Routes

In [3]:
route_dirs = sorted(
    path for path in DATASET_ROOT.glob("driver_*/route_*")
    if path.is_dir()
)

route_inventory = pd.DataFrame(
    {
        "route_id": [f"{path.parent.name}/{path.name}" for path in route_dirs],
        "driver_id": [path.parent.name for path in route_dirs],
        "route_name": [path.name for path in route_dirs],
        "route_path": [str(path) for path in route_dirs],
    }
)

present_routes = set(route_inventory["route_id"])

print(f"Discovered {len(route_inventory)} local routes.")
display(route_inventory.head(10))

Discovered 25 local routes.


,route_id,driver_id,route_name,route_path
0,driver_54/route_1,driver_54,route_1,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
1,driver_97/route_1,driver_97,route_1,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
2,driver_97/route_10,driver_97,route_10,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
3,driver_97/route_11,driver_97,route_11,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
4,driver_97/route_12,driver_97,route_12,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
5,driver_97/route_13,driver_97,route_13,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
6,driver_97/route_14,driver_97,route_14,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
7,driver_97/route_15,driver_97,route_15,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
8,driver_97/route_16,driver_97,route_16,/Users/mbaeuerl/coding/ComputationalModelsSoSe...
9,driver_97/route_17,driver_97,route_17,/Users/mbaeuerl/coding/ComputationalModelsSoSe...


## Import All Route CSVs

`routes` preserves each route separately:

```python
routes["driver_97/route_1"]["data"]["vehicle_dynamics"]
```

`data_by_modality` concatenates the same modality across all present routes:

```python
data_by_modality["vehicle_dynamics"]
```

In [4]:
def read_route_metadata(route_path: Path) -> dict:
    """Read metadata.json for a route if present."""
    metadata_path = route_path / "metadata.json"
    if not metadata_path.exists():
        return {}
    with metadata_path.open("r") as file:
        return json.load(file)


def read_route_csv(route_path: Path, file_name: str, route_id: str, driver_id: str, route_name: str) -> pd.DataFrame | None:
    """Read one route CSV and add route identity columns."""
    csv_path = route_path / file_name
    if not csv_path.exists():
        return None

    frame = pd.read_csv(csv_path, low_memory=False)
    frame.insert(0, "route_id", route_id)
    frame.insert(1, "driver_id", driver_id)
    frame.insert(2, "route_name", route_name)
    return frame


routes = {}

for row in route_inventory.itertuples(index=False):
    route_path = Path(row.route_path)
    route_data = {}

    for modality, file_name in ROUTE_FILES.items():
        frame = read_route_csv(
            route_path=route_path,
            file_name=file_name,
            route_id=row.route_id,
            driver_id=row.driver_id,
            route_name=row.route_name,
        )
        if frame is not None:
            route_data[modality] = frame

    routes[row.route_id] = {
        "path": route_path,
        "metadata": read_route_metadata(route_path),
        "data": route_data,
    }

data_by_modality = {}

for modality in ROUTE_FILES:
    frames = [
        route_bundle["data"][modality]
        for route_bundle in routes.values()
        if modality in route_bundle["data"]
    ]
    if frames:
        data_by_modality[modality] = pd.concat(frames, ignore_index=True)

modality_summary = pd.DataFrame(
    [
        {
            "modality": modality,
            "routes_loaded": frame["route_id"].nunique(),
            "rows": len(frame),
            "columns": frame.shape[1],
        }
        for modality, frame in data_by_modality.items()
    ]
).sort_values("modality")

display(modality_summary)

,modality,routes_loaded,rows,columns
1,driver_state,25,1136726,28
0,gps,25,568374,24
3,imu,25,5683519,10
4,localization,25,1136726,39
6,planning,25,1136726,17
2,radar,25,1136726,20
5,vehicle_dynamics,25,5683519,40


## Preview Imported Data

In [5]:
for modality, frame in data_by_modality.items():
    print(f"\n{modality}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")
    display(frame.head(3))


gps: 568,374 rows x 24 columns


,route_id,driver_id,route_name,time_s,gps_lat,gps_lon,gps_alt,gps_speed,gps_bearing,gps_hAcc,gps_vAcc,gps_speedAcc,gps_bearingAcc,gps_unixTimestamp,gps_hasFix,gps_source,gps_flags,gps_phone_lat,gps_phone_lon,gps_phone_alt,gps_phone_speed,gps_phone_bearing,gps_phone_hAcc,gps_phone_hasFix
0,driver_54/route_1,driver_54,route_1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.492291,-81.429312,5.093687,2.104912,137.509689,0.0,1
1,driver_54/route_1,driver_54,route_1,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.492291,-81.429312,5.093687,2.104912,137.509689,0.0,1
2,driver_54/route_1,driver_54,route_1,0.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.492291,-81.429312,5.093687,2.104912,137.509689,0.0,1



driver_state: 1,136,726 rows x 28 columns


,route_id,driver_id,route_name,time_s,face_yaw,face_pitch,face_roll,face_pos_x,face_pos_y,faceProb,leftEyeProb,rightEyeProb,leftBlinkProb,rightBlinkProb,sunglassesProb,occludedProb,readyProb_0,readyProb_1,readyProb_2,readyProb_3,notReadyProb_0,notReadyProb_1,poorVisionProb,awarenessStatus,isDistracted,distractedType,faceDetected,isRHD
0,driver_54/route_1,driver_54,route_1,0.00,0.127837,0.34516,-0.025567,0.191756,0.012784,0.109435,0.274569,0.264503,0.295402,0.284871,0.004636,0.487219,0.563573,0.487219,0.449043,0.449043,0.143107,0.525545,0.943343,1.0,0,0,0,0
1,driver_54/route_1,driver_54,route_1,0.05,0.127837,0.34516,-0.025567,0.191756,0.012784,0.109435,0.274569,0.264503,0.295402,0.284871,0.004636,0.487219,0.563573,0.487219,0.449043,0.449043,0.143107,0.525545,0.943343,1.0,0,0,0,0
2,driver_54/route_1,driver_54,route_1,0.10,0.127837,0.34516,-0.025567,0.191756,0.012784,0.109435,0.274569,0.264503,0.295402,0.284871,0.004636,0.487219,0.563573,0.487219,0.449043,0.449043,0.143107,0.525545,0.943343,1.0,0,0,0,0



radar: 1,136,726 rows x 20 columns


...

,route_id,driver_id,route_name,time_s,cam_trans_x,cam_trans_y,cam_trans_z,cam_rot_x,cam_rot_y,cam_rot_z,rpyCalib_roll,rpyCalib_pitch,rpyCalib_yaw,calStatus,calPerc,lp_steerRatio,lp_stiffnessFactor,lp_angleOffsetDeg,lp_angleOffsetAverageDeg,lt_latAccelFactor,lt_frictionCoeff,geo_lat,geo_lon,geo_alt,vel_n,vel_e,vel_d,orient_roll,orient_pitch,orient_yaw,gpsOK,vel_x,vel_y,vel_z,acc_dev_x,acc_dev_y,acc_dev_z,pose_inputsOK,pose_posenetOK
0,driver_54/route_1,driver_54,route_1,0.00,2.232808,-0.013266,0.003342,0.000131,0.000176,0.000546,-0.000004,0.004629,-0.005799,calibrated,100.0,12.07206,1.0,-0.412402,-0.412402,0.0,0.0,36.374079,-40.976167,-2854.79148,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,driver_54/route_1,driver_54,route_1,0.05,2.232808,-0.013266,0.003342,0.000131,0.000176,0.000546,-0.000004,0.004629,-0.005799,calibrated,100.0,12.07206,1.0,-0.412402,-0.412402,0.0,0.0,36.374079,-40.976167,-2854.79148,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,driver_54/route_1,driver_54,route_1,0.10,2.232808,-0.013266,0.003342,0.000131,0.000176,0.000546,-0.000004,0.004629,-0.005799,calibrated,100.0,12.07206,1.0,-0.412402,-0.412402,0.0,0.0,36.374079,-40.976167,-2854.79148,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



vehicle_dynamics: 5,683,519 rows x 40 columns


,route_id,driver_id,route_name,time_s,vEgo,aEgo,vEgoRaw,standstill,steeringAngleDeg,steeringTorque,steeringPressed,gas,gasPressed,brake,brakePressed,cruiseState_enabled,cruiseState_available,cruiseState_speed,cc_enabled,cc_latActive,cc_longActive,leftBlinker,rightBlinker,actuators_accel,actuators_torque,actuators_speed,actuators_curvature,co_accel,co_brake,co_gas,co_steer,co_steeringAngleDeg,co_curvature,cs_enabled,cs_active,cs_curvature,cs_desiredCurvature,cs_vCruise,cs_longControlState,cs_forceDecel
0,driver_54/route_1,driver_54,route_1,0.00,0.0,0.0,0.0,1,3.4,1.99,1,0.0,0,0.0,0,0,0,4.4704,0,0,0,0,0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,-0.297465,0.0,255.0,off,0
1,driver_54/route_1,driver_54,route_1,0.01,0.0,0.0,0.0,1,3.4,1.99,1,0.0,0,0.0,0,0,0,4.4704,0,0,0,0,0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,-0.297465,0.0,255.0,off,0
2,driver_54/route_1,driver_54,route_1,0.02,0.0,0.0,0.0,1,3.4,1.99,1,0.0,0,0.0,0,0,0,4.4704,0,0,0,0,0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,-0.297465,0.0,255.0,off,0



planning: 1,136,726 rows x 17 columns


,route_id,driver_id,route_name,time_s,model_desiredCurvature,model_desiredAcceleration,laneLeft_prob,laneRight_prob,laneLeft_y,laneRight_y,laneChangeState,laneChangeDirection,aTarget,shouldStop,hasLead,fcw,longPlanSource
0,driver_54/route_1,driver_54,route_1,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.708904,0,0,0,cruise
1,driver_54/route_1,driver_54,route_1,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.708904,0,0,0,cruise
2,driver_54/route_1,driver_54,route_1,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.708904,0,0,0,cruise


## Import Takeover Events

In [8]:
takeover_events = pd.read_csv(TAKEOVER_EVENTS_PATH, low_memory=False)

print(f"Loaded {len(takeover_events):,} takeover events from benchmark table.")
display(takeover_events.head())

Loaded 1,432 takeover events from benchmark table.


,event_id,route_id,driver_id,vehicle_model,event_time_sec,pre_context_sec,post_context_sec,event_type,definition_version
0,TAKE_00001,driver_98/route_2,driver_98,ACURA_MDX_3G_MMR,5.05,5.0,6129.4,takeover,or_v1
1,TAKE_00002,driver_98/route_2,driver_98,ACURA_MDX_3G_MMR,43.03,43.0,6091.5,takeover,or_v1
2,TAKE_00003,driver_98/route_2,driver_98,ACURA_MDX_3G_MMR,105.93,105.9,6028.6,takeover,or_v1
3,TAKE_00004,driver_98/route_2,driver_98,ACURA_MDX_3G_MMR,178.41,178.4,5956.1,takeover,or_v1
4,TAKE_00005,driver_98/route_2,driver_98,ACURA_MDX_3G_MMR,1266.54,1266.5,4867.9,takeover,or_v1


## Filter Takeover Events to Routes Present Locally

In [13]:
takeover_events_present = takeover_events[
    takeover_events["route_id"].isin(present_routes)
].copy()

takeover_events_missing_routes = takeover_events[
    ~takeover_events["route_id"].isin(present_routes)
].copy()

event_route_summary = pd.DataFrame(
    {
        "metric": [
            "local routes present",
            "benchmark takeover events total",
            "takeover events on local routes",
            "takeover events on missing routes",
            "local routes with takeover events",
        ],
        "value": [
            len(present_routes),
            len(takeover_events),
            len(takeover_events_present),
            len(takeover_events_missing_routes),
            takeover_events_present["route_id"].nunique(),
        ],
    }
)

display(sorted(list(present_routes)))

display(event_route_summary)
display(takeover_events_present.head(10))

['driver_54/route_1',
 'driver_97/route_1',
 'driver_97/route_10',
 'driver_97/route_11',
 'driver_97/route_12',
 'driver_97/route_13',
 'driver_97/route_14',
 'driver_97/route_15',
 'driver_97/route_16',
 'driver_97/route_17',
 'driver_97/route_18',
 'driver_97/route_19',
 'driver_97/route_2',
 'driver_97/route_20',
 'driver_97/route_21',
 'driver_97/route_22',
 'driver_97/route_23',
 'driver_97/route_29',
 'driver_97/route_3',
 'driver_97/route_4',
 'driver_97/route_5',
 'driver_97/route_6',
 'driver_97/route_7',
 'driver_97/route_8',
 'driver_97/route_9']

,metric,value
0,local routes present,25
1,benchmark takeover events total,1432
2,takeover events on local routes,32
3,takeover events on missing routes,1400
4,local routes with takeover events,5


,event_id,route_id,driver_id,vehicle_model,event_time_sec,pre_context_sec,post_context_sec,event_type,definition_version
151,TAKE_00152,driver_97/route_21,driver_97,HONDA_CIVIC,4544.78,4544.8,860.4,takeover,or_v1
152,TAKE_00153,driver_97/route_21,driver_97,HONDA_CIVIC,4576.78,4576.8,828.4,takeover,or_v1
153,TAKE_00154,driver_97/route_21,driver_97,HONDA_CIVIC,4650.08,4650.1,755.1,takeover,or_v1
154,TAKE_00155,driver_97/route_21,driver_97,HONDA_CIVIC,4693.71,4693.7,711.5,takeover,or_v1
155,TAKE_00156,driver_97/route_21,driver_97,HONDA_CIVIC,4783.00,4783.0,622.2,takeover,or_v1
156,TAKE_00157,driver_97/route_21,driver_97,HONDA_CIVIC,4902.92,4902.9,502.3,takeover,or_v1
157,TAKE_00158,driver_97/route_21,driver_97,HONDA_CIVIC,5056.02,5056.0,349.2,takeover,or_v1
158,TAKE_00159,driver_97/route_21,driver_97,HONDA_CIVIC,5150.40,5150.4,254.8,takeover,or_v1
159,TAKE_00160,driver_97/route_21,driver_97,HONDA_CIVIC,5247.75,5247.8,157.4,takeover,or_v1
160,TAKE_00161,driver_97/route_22,driver_97,HONDA_CIVIC,612.93,612.9,4114.2,takeover,or_v1


## Optional Checks

In [ ]:
events_by_local_route = (
    takeover_events_present
    .groupby("route_id", as_index=False)
    .agg(
        takeover_events=("event_id", "count"),
        first_event_time_sec=("event_time_sec", "min"),
        last_event_time_sec=("event_time_sec", "max"),
    )
    .sort_values(["takeover_events", "route_id"], ascending=[False, True])
)

display(events_by_local_route)

local_routes_without_takeover_events = sorted(present_routes - set(takeover_events_present["route_id"]))
print(f"Local routes without benchmark takeover events: {len(local_routes_without_takeover_events)}")
local_routes_without_takeover_events[:20]

## Optional Save

Uncomment this cell if you want a CSV copy of the filtered takeover events.

In [ ]:
# OUTPUT_DIR = PROJECT_ROOT / "reports" / "data_import"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# filtered_events_path = OUTPUT_DIR / "takeover_events_present_routes.csv"
# takeover_events_present.to_csv(filtered_events_path, index=False)
# filtered_events_path